In [ ]:
! git clone https://github.com/mmeagher/point-e

In [ ]:
%cd point-e/

In [ ]:
pip install -e .

In [ ]:
import torch
from tqdm.auto import tqdm

import point_e.util.point_cloud as pt

from point_e.diffusion.configs import DIFFUSION_CONFIGS, diffusion_from_config
from point_e.diffusion.sampler import PointCloudSampler
from point_e.models.download import load_checkpoint
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.plotting import plot_point_cloud

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('creating base model...')
base_name = 'base40M-textvec'
base_model = model_from_config(MODEL_CONFIGS[base_name], device)
base_model.eval()
base_diffusion = diffusion_from_config(DIFFUSION_CONFIGS[base_name])

print('creating upsample model...')
upsampler_model = model_from_config(MODEL_CONFIGS['upsample'], device)
upsampler_model.eval()
upsampler_diffusion = diffusion_from_config(DIFFUSION_CONFIGS['upsample'])

print('downloading base checkpoint...')
base_model.load_state_dict(load_checkpoint(base_name, device))

print('downloading upsampler checkpoint...')
upsampler_model.load_state_dict(load_checkpoint('upsample', device))

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import os

# Create output directory for point clouds
output_dir = 'pointcloud_variations'
os.makedirs(output_dir, exist_ok=True)

# Define prompt variations
prompts = [
    'furniture for dreaming whose surface and form is shaped by the emotion of fear, sadness and anxiety and which involves loved ones in a setting that is very real',
    'a surface for dreaming whose form is shaped by the emotion of fear, sadness and anxiety and which involves loved ones in a setting that is very real',
    'a surface for dreaming whose form is shaped by the emotion of gladness and delight and which involves pets in a setting that is unfamiliar and unreal'
]

# Define parameter ranges for variations
guidance_scales = [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
karras_steps_options = [64, 96, 128]
s_churn_options = [0.5, 1.0, 2.0, 3.0, 5.0]

# Initialize tracking dataframe
tracking_data = []

print(f'Setup complete. Will generate 100 variations.')

In [ ]:
# Generate 10 variations
num_variations = 10

print(f'Starting generation of {num_variations} variations...')
print(f'This may take a while. Progress will be shown below.')

for i in range(num_variations):
    variation_id = i + 1

    # Select parameters for this variation
    prompt = prompts[i % len(prompts)]
    guidance_scale = guidance_scales[i % len(guidance_scales)]
    karras_steps = karras_steps_options[(i // len(prompts)) % len(karras_steps_options)]
    s_churn_base = s_churn_options[(i // (len(prompts) * len(karras_steps_options))) % len(s_churn_options)]
    s_churn_upsample = s_churn_options[(i // len(guidance_scales)) % len(s_churn_options)] * 0.2

    print(f'\n--- Variation {variation_id}/{num_variations} ---')
    print(f'Prompt: {prompt[:60]}...' if len(prompt) > 60 else f'Prompt: {prompt}')
    print(f'Parameters: guidance={guidance_scale}, steps={karras_steps}, churn={s_churn_base:.1f}')

    # Create sampler with these parameters
    sampler = PointCloudSampler(
        device=device,
        models=[base_model, upsampler_model],
        diffusions=[base_diffusion, upsampler_diffusion],
        num_points=[1024, 4096 - 1024],
        aux_channels=['R', 'G', 'B'],
        guidance_scale=[guidance_scale, 0.0],
        model_kwargs_key_filter=('texts', ''),
        karras_steps=[karras_steps, karras_steps],
        s_churn=[s_churn_base, s_churn_upsample],
    )

    # Generate point cloud
    samples = None
    for x in tqdm(sampler.sample_batch_progressive(batch_size=1, model_kwargs=dict(texts=[prompt])),
                  desc=f'Generating {variation_id}'):
        samples = x

    # Convert to point cloud
    pc = sampler.output_to_point_clouds(samples)[0]

    # Apply color based on height for consistency
    z_coords = pc.coords[:, 2]
    z_min, z_max = z_coords.min(), z_coords.max()
    if z_max > z_min:
        normalized_z = (z_coords - z_min) / (z_max - z_min)
    else:
        normalized_z = np.zeros_like(z_coords)

    pc.channels['R'] = normalized_z
    pc.channels['G'] = 0.3 * np.ones_like(normalized_z)
    pc.channels['B'] = 1.0 - normalized_z

    # Save point cloud
    filename = f'pointcloud_{variation_id:03d}.ply'
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'wb') as f:
        pc.write_ply(f)

    # Record parameters
    tracking_data.append({
        'id': variation_id,
        'prompt': prompt,
        'guidance_scale_base': guidance_scale,
        'guidance_scale_upsample': 0.0,
        'karras_steps_base': karras_steps,
        'karras_steps_upsample': karras_steps,
        's_churn_base': s_churn_base,
        's_churn_upsample': s_churn_upsample,
        'num_points': 4096,
        'filename': filename,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

    print(f'Saved: {filepath}')

print(f'\n✓ All {num_variations} variations generated successfully!')

In [ ]:
# Save tracking data to CSV
df = pd.DataFrame(tracking_data)
csv_filename = 'pointcloud_parameters.csv'
df.to_csv(csv_filename, index=False)

print(f'Parameter tracking saved to: {csv_filename}')
print(f'\nFirst 10 variations:')
print(df.head(10).to_string())
print(f'\n...')
print(f'\nLast 10 variations:')
print(df.tail(10).to_string())
print(f'\nTotal variations: {len(df)}')

In [ ]:
# Optional: Visualize a specific variation
# Change the variation_id below to view different results
variation_to_view = 1  # Change this to view different variations (1-100)

if variation_to_view <= len(tracking_data):
    # Load the saved point cloud
    variation_file = os.path.join(output_dir, f'pointcloud_{variation_to_view:03d}.ply')

    if os.path.exists(variation_file):
        # Get parameters for this variation
        params = df[df['id'] == variation_to_view].iloc[0]

        print(f'Viewing Variation {variation_to_view}:')
        print(f'Prompt: {params["prompt"]}')
        print(f'Guidance Scale: {params["guidance_scale_base"]}')
        print(f'Karras Steps: {params["karras_steps_base"]}')
        print(f'S-Churn: {params["s_churn_base"]}')
        print(f'File: {params["filename"]}')

        # Note: To view the point cloud, you would need to load it from the PLY file
        # For now, this cell just displays the parameters
        print(f'\nPoint cloud saved at: {variation_file}')
else:
    print(f'Variation {variation_to_view} not found. Valid range: 1-{len(tracking_data)}')